# 01. Khảo sát dữ liệu

Mục tiêu của notebook này là xác nhận cấu trúc thực tế của file tải từ Kaggle:
số dòng, số cột, kiểu dữ liệu, tỷ lệ giá trị thiếu, và số mức của các biến định
tính. Con số quan trọng nhất cần rút ra là số thuộc tính dự kiến sau khi mã hóa
one-hot, vì đề bài yêu cầu bài toán đủ lớn về số thuộc tính.

Đặt file csv vào `data/raw/` rồi sửa biến `CSV_PATH` bên dưới.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.plotting import set_style

set_style()


In [ ]:
CSV_PATH = ROOT / "data" / "raw" / "used_phone_price_prediction_1M.csv"

df = pd.read_csv(CSV_PATH)
print(f"{df.shape[0]} dòng, {df.shape[1]} cột")
df.head()

## Kiểu dữ liệu và giá trị thiếu

In [ ]:
info = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(),
        "missing_ratio": df.isna().mean().round(4),
    }
)
info.sort_values("missing_ratio", ascending=False)

## Biến mục tiêu

Xác nhận cột nào là giá bán lại, và xem phân phối của nó có lệch phải hay không.
Nếu lệch mạnh thì sẽ dùng `log(giá)` làm biến mục tiêu ở notebook 02.

In [ ]:
from src.data import guess_target

TARGET = guess_target(df)
print("cột mục tiêu:", TARGET)
df[TARGET].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(df[TARGET].dropna(), bins=50, color="#1f77b4")
axes[0].set_title("Target")
axes[0].set_xlabel(TARGET)

positive = df[TARGET].dropna()
positive = positive[positive > 0]
axes[1].hist(np.log(positive), bins=50, color="#2ca02c")
axes[1].set_title("log(Target)")
axes[1].set_xlabel(f"log({TARGET})")
fig.tight_layout()

## Số thuộc tính dự kiến sau khi mã hóa

Ước lượng bề rộng của ma trận thiết kế trước khi thực sự xây dựng nó. Nếu con số
này quá nhỏ, cần xem lại phương án ở mục 3.3 của kế hoạch: giữ cột `model` với
nhiều mức hơn, hoặc bổ sung biến tương tác.

In [ ]:
from src.data import coerce_numeric_like, split_column_types

converted = coerce_numeric_like(df)
numeric, categorical = split_column_types(converted, TARGET)

print("cột định lượng:", len(numeric), numeric)
print("cột định tính :", len(categorical))

rows = []
for col in categorical:
    counts = converted[col].astype(str).value_counts()
    rows.append(
        {
            "column": col,
            "n_levels": counts.size,
            "n_levels_ge_10": int((counts >= 10).sum()),
        }
    )
levels = pd.DataFrame(rows).sort_values("n_levels", ascending=False)
estimated_d = len(numeric) + int((levels["n_levels_ge_10"] + 1 - 1).sum())
print(f"\nsố thuộc tính ước lượng sau one-hot (gộp mức hiếm): khoảng {estimated_d}")
levels

## Ghi chú cho báo cáo

Điền vào đây các con số cần trích dẫn ở mục 1 của báo cáo: số dòng, số cột thô,
cột mục tiêu, và số thuộc tính dự kiến.